# 05 · Prospective discovery screen (blood traits, in-domain)

What the tool is *for*: screening noncoding variants for an interpretable, uncharacterized,
**falsifiable** regulatory hypothesis — **in-domain only** (blood traits in K562, where the
engine is validated 0.716). The pipeline selects via the quadrant (large `|Δ|` + concordant
motif + GWAS trait + sparse literature); we take what surfaces and **verify novelty by
hand**. See `docs/discovery_worked_example_rs342293.md` for how the guardrail caught a
false-novelty last pass.

In [ ]:
# --- Setup: GPU + install RegLens -------------------------------------------
# Upload reglens_for_colab.zip via the Files panel first (git archive of the repo).
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > GPU.'
!pip -q install 'tensorflow>=2.12' pyfaidx typer numpy matplotlib
!unzip -oq /content/reglens_for_colab.zip -d /content/RegLens && pip -q install -e /content/RegLens
import os; os.chdir('/content/RegLens'); print('cwd', os.getcwd())

In [ ]:
# --- Reference genome (hg38) + pretrained K562 ChromBPNet model -------------
import glob, os
os.makedirs('/content/ref', exist_ok=True)
%cd /content/ref
# hg38 via the Broad public bucket: resolves on Colab, chr-named, ships its .fai.
!wget -c -O hg38.fa     https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta
!wget -c -O hg38.fa.fai https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta.fai
# ENCODE K562 ATAC ChromBPNet models (5 folds, bias-corrected).
![ -f ENCFF984RAF.tar.gz ] || wget -q -c -O ENCFF984RAF.tar.gz https://www.encodeproject.org/files/ENCFF984RAF/@@download/ENCFF984RAF.tar.gz
!mkdir -p encode_models && tar -xzf ENCFF984RAF.tar.gz -C encode_models
n = len(glob.glob('encode_models/**/*chrombpnet_nobias*.h5', recursive=True))
print(n, 'K562 folds | genome:', os.path.exists('hg38.fa'))
%cd /content/RegLens

In [ ]:
# --- Reasoning layer: Anthropic SDK + API key -------------------------------
!pip -q install anthropic
import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'   # <-- paste your key
assert os.environ['ANTHROPIC_API_KEY'].startswith('sk-'), 'set your key above'

In [ ]:
# Fold + reverse-complement averaged K562 backend (the validated engine).
from reglens.tools.chrombpnet_score import ChromBPNetScorer, load_backend
scorer = ChromBPNetScorer(load_backend('/content/ref/encode_models', average_rc=True),
                          window_length=2114, model_name='K562-5fold+rc')
GENOME = '/content/ref/hg38.fa'

In [ ]:
# --- Screen an UNBIASED pool pulled from the GWAS Catalog --------------------
from reglens.validation import discovery as dsc
from reglens.agents.multi_agent import MultiAgentInterpreter

pool, seen = [], set()
for trait in ['platelet count', 'erythrocyte count', 'hemoglobin measurement',
              'reticulocyte count', 'neutrophil count']:
    for c in dsc.fetch_gwas_variants(trait, max_variants=20, seed=7):
        if c.rsid not in seen:
            seen.add(c.rsid); pool.append(c)
print(f'screening {len(pool)} unbiased blood-trait variants')

scored = dsc.run_discovery_screen(pool, scorer=scorer, genome_path=GENOME,
                                  celltype='K562', progress=True)
print(dsc.render_screen(scored))

In [ ]:
# --- Interpret a genuine quadrant survivor (only if one clears the bar) ------
quadrant = [(s, b) for s, b in scored if s.in_quadrant]
if quadrant:
    s, b = quadrant[0]
    res = MultiAgentInterpreter().deliberate(b)
    print(s.rsid, s.gene, s.motif_tf, '| conf', res.interpretation.confidence)
    print(res.interpretation.mechanism)
    print('\nNEXT: verify novelty BY HAND (PubMed/Scholar/Open Targets) before any claim.')
else:
    print('No quadrant hit. Do NOT lower the bar. Expand the pool or report the null.')

## Honest reading

Last pass: 0 quadrant hits across 100 unbiased GWAS variants (raw leads are LD tags that
don't fire the engine), and the top sparse-literature lead (rs342293) turned out to be
**already characterized** on manual check — the guardrail working. Report the screen
outcome; never manufacture a claim.